<img src="https://www.digi-hand.de/wp-content/themes/hitchcock/images/logo-tu-iwf.png" align='right' width="20%">
<a name="divtop"></a> 

# Pipeline für die Datenaufbereitung
                                                   
——————————————————————————————————————————————————————————————————————————————————————————————————————————————————————

**Autor**: Gefertec Gruppe_WS22 | **Letztes Update**: 24. Mai 2023

### Notebook-Ziele
<div class="alert alert-block" style="background-color: #90EE90">
<font color=green>
    
In Zusammenarbeit mit der Pipeline für die Datenaufbereitung mit assistive labelling werden wir:

- Read Bilder aus Videos und

- Data Clean
   
</font> 
    
</div>

## Inhaltsverzeichnis

* [Einführung](#introduction)
* [Importe](#imports)
* [Komfortfunktionen](#function)
    - [read step](#read)
    - [data clean](#clean)
* [Methode: grafBiTree](#tree)
* [Die gesamte Pipeline ausführen ](#pipeline)
    - [read step](#step1)
    - [data clean step ](#step2)
* [Datensatz](#datensatz)
* [Resultat](#resultat)

<a name="introduction"></a>
## Einführung

Dieses Notebook behandelt die Datenvorverarbeitungspipeline für das Projekt von Gefertec, die auch die automatische “assistive labelling” abdeckt.

Installation: Sie benötigen die Installation von OpenCV 3.2, Programmierungsprache(OpenCV-Python), IDE(Anaconda).

Ziel von "Video zu Bild": 
> 1. Zusammenhängendem Bilde
2. Durch einzelnen Frame extrahieren
3. Eigentlich Frame überprüfen

Ziel von "Data Clean": 
> 1. Automatisiert(Integriertes Modell)
2. Überlichtet; Unterlichtet; Unsauber Bild(Lokale Überbelichtung, Temperaturrausch und niedriger Kontrast usw.)

## Imports <a name="imports"></a>
Für dieses Notebook erforderliche Pakete importieren:

In [ ]:
import cv2
import glob

# Numpy library:
import numpy as np

#Modify the path to a directory on your machine
import os

import matplotlib.pyplot as plt
from PIL import Image,ImageChops
import random
from typing import List
from tqdm import tqdm

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from scipy.stats import entropy
from math import log, e
import pandas as pd

[Top of Notebook](#divtop)

## Komfortfunktionen<a name="function"></a>
Hier definieren wir einige Funktionen und einige Parameter, die wir im gesamten Notebook wiederholt verwenden werden.

In [7]:
# Um alles einfacher zu machen, werden alle Dateien von der Pipeline gespeichert
# und Pipelineschritte werden im Arbeitsverzeichnis gespeichert.
# Dies wird für die Pipelines der Ebenen 2 und 3 wichtiger sein
# Sobald wir mit Assoziationsdateien arbeiten.

### read step<a name="read"></a>

In [ ]:
def save_all_frames(video_path, dir_path, basename, ext='jpg'):
    """This function also removes the duplicate frames.
    
    From: https://note.nkmk.me/en/python-opencv-video-to-still-image/
    """
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return

    os.makedirs(dir_path, exist_ok=True)
    base_path = os.path.join(dir_path, basename)

    digit = len(str(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))))

    n = 0

    while True:
        ret, frame = cap.read()
        if ret:
            cv2.imwrite('{}_{}.{}'.format(base_path, str(n).zfill(digit), ext), frame)
            n += 1
        else:
            return

In [ ]:
def getVideoToImage(file_name_pattern, video_files):
    """
    To convert video to image, you need to run the function “save_all_frames first”
    """
    for video_file in video_files:
        head, sep, tail = video_file.partition("Sample")
        frame_base_name = sep + tail
        frame_base_name = frame_base_name.rstrip(".avi").replace("/", "_")
        save_all_frames(video_file, './extracted_frames_no_duplicates', frame_base_name)

In [ ]:
def get_duration_from_cv2(filename):
    """
    Funktion zur Rechen der Videoszeit:
    """
    cap = cv2.VideoCapture(filename)
    if cap. isOpened():
        rate = cap.get(5)
        frame_num=cap.get(7)
        duration = frame_num/rate
        return duration
    return -1


In [ ]:
def get_number_from_image(file_name_pattern):
    """
    Funktion zur Rechen der Videoszeit:
    """
    video_files = glob.glob(file_name_pattern, recursive=True)
    num_png = len(video_files)
    print("Eigentlich frames : {0}".format(num_png))
    return num_png

### Data Clean step <a name="clean"></a>

In [ ]:
def is_greyscale(im):
    """
    Check if image is monochrome (1 channel or 3 identical channels)

    From Stackoverflow: https://stackoverflow.com/questions/23660929/how-to-check-whether-a-jpeg-image-is-color-or-gray-scale-using-only-python-stdli
    """
    if im.mode not in ("L", "RGB"):
        raise ValueError("Unsuported image mode")

    if im.mode == "RGB":
        rgb = im.split()
        if ImageChops.difference(rgb[0],rgb[1]).getextrema()[1]!=0: 
            return False
        if ImageChops.difference(rgb[0],rgb[2]).getextrema()[1]!=0: 
            return False
    return True

In [ ]:
def show_histograms(path: str, image_nums: List[int]):
    """
    From: https://stackoverflow.com/questions/22159160/python-calculate-histogram-of-image
    """
    fig, axes = plt.subplots(1,len(image_nums))
    fig.set_size_inches(15,5)
    for image_num, ax in zip(image_nums, axes):
        image_path = f"{path}_{str(image_num).zfill(4)}.jpg"
        im = cv2.imread(image_path)
        # calculate mean value from RGB channels and flatten to 1D array
        vals = im.mean(axis=2).flatten()
        # plot histogram with 255 bins
        b, bins, patches = ax.hist(vals, 255)
        ax.set_xlim([0,255])

    plt.show()

In [ ]:
def get_box_plot_data(labels, bp):
    """
    From: https://stackoverflow.com/questions/23461713/obtaining-values-used-in-boxplot-using-python-and-matplotlib
    """
    rows_list = []
    for i in range(len(labels)):
        dict1 = {}
        dict1['label'] = labels[i]
        dict1['lower_whisker'] = bp['whiskers'][i*2].get_ydata()[1]
        dict1['lower_quartile'] = bp['boxes'][i].get_ydata()[1]
        dict1['median'] = bp['medians'][i].get_ydata()[1]
        dict1['upper_quartile'] = bp['boxes'][i].get_ydata()[2]
        dict1['upper_whisker'] = bp['whiskers'][(i*2)+1].get_ydata()[1]
        rows_list.append(dict1)

    return pd.DataFrame(rows_list)

In [ ]:
def is_overexposed(image, bins: int = 5):
    """
    We take a look at the histogram of the flattened picture (which is converted to greyscale first by taking the mean across the three color channels).
    Only if the last bin contains the largest number of pixels, we define the image to be overexposed.
    """
    return np.argmax(np.histogram(image, bins=bins)[0]) == bins - 1

In [ ]:
def is_underexposed(varianz, schwellenwert_varianz):
    """
    Only when the grayscale median value is less than the underexposure threshold and the variance threshold, it is considered underexposed
    """
    return varianz < schwellenwert_varianz

In [ ]:
def high_smoke(image):      # Prozentsatz der hellen Pixelwerte im gesamten Bild
    '''
    Gaswolke (Rauchwolke mit hoher Intensität) 0 Pixelwert-> nicht Wolke, 1 Pixelwert-> Wolke
    '''
    th1, dst1 = cv2.threshold(image, 200, 255, cv2.THRESH_BINARY)
    th2, dst2 = cv2.threshold(image, 155, 255, cv2.THRESH_BINARY)
    dst = dst2 - dst1
    # Wolke normal legt oben der Schmelzbad, um genauer Ergebnisse:
    dst = dst[:700, :] 
    kernel = np. ones ((4,4) ,np.uint8)
    dilate = cv2.dilate(dst, kernel)
    closing = cv2. erode(dilate, kernel)
    # Prozent rechnen:
    gesamtpixel = np.shape(closing)[0]*np.shape(closing)[1]
    return np.sum((np.histogram(closing,bins=2)[0])[1])/gesamtpixel

In [ ]:
def mid_smoke(image):      # Prozentsatz der hellen Pixelwerte im gesamten Bild
    '''
    Gaswolke (Rauchwolke mit hoher Intensität) 0 Pixelwert-> nicht Wolke, 1 Pixelwert-> Wolke
    '''
    th1, dst1 = cv2.threshold(image, 200, 255, cv2.THRESH_BINARY)
    th2, dst2 = cv2.threshold(image, 100, 255, cv2.THRESH_BINARY)
    dst = dst2- dst1
    # Wolke normal legt oben der Schmelzbad, um genauer Ergebnisse. Verbeserrung der Ergebnisse durch Closeing
    dst = dst[:700, :]
    kernel = np. ones ((4,4) ,np.uint8)
    dilate = cv2.dilate(dst, kernel)
    closing = cv2. erode(dilate, kernel)
    # Prozent rechnen:
    gesamtpixel = np.shape(closing)[0]*np.shape(closing)[1]
    return np.sum((np.histogram(closing,bins=2)[0])[1])/gesamtpixel

[Top of Notebook](#divtop)

<a name="tree"></a>
## Methode: grafBiTree
Detaillierte methodische Informationen finden Sie im Artikel.


#### Definition
Bilder scheinen nicht genau Graustufen zu sein und anhand von Histogrammen weniger gut erkennen. Viele dunkle Bilder, aber kaum welche die komplett “schwarz” sind

Klassen | Verarbeitungsverfahren| Beschreibung  | Schwierigkeiten
---| ---| ---|---|
1.Mit Lichtbogen  |Beibehalten| Lichtbogen,Schweißbad, die erkennbar | **Schwierig** von Bildern zu unterscheiden, die von Gaswolken verdeckt werden 
2.Ohne Lichtbogen |Beibehalten|Nur Schweißbad erkennbar | **Schwierig** von unterbelichteten Bildern zu unterscheiden
3.Überlichtete    |Löschen    |Ganz Bild weiß sein      | Überbelichtete Bilder lassen sich anhand **von Histogrammen gut** filtern
4.Unterlichtete   |Löschen    |Ganz Bild schwarz sein   | Gleich wie Ohne Lichtbogen
5.Unsauber  Bilder|Löschen    |Gasvolke, Bildrauschen, Lokale Überlichtung| Gleich wie Mit Lichtbogen

#### Zusätzliche Parameter und Auswählen der Schwellenwert
Die Hauptquelle der aktuellen Analyse ist das Histogramm des Bildes. Gegenwärtig ist es schwierig, mindestens fünf Arten von Bildern durch einen einzigen Parameter (den Durchschnittswert des Histogramms) zu unterscheiden. Es werden also neue Parameter eingeführt. Dann werden weitere Informationen basierend auf dem Histogramm erweitert. Wie Beispiel: Varianz, Entropie.

> **Mittelwert:** Der mittlere Grauwert des Bildes.

> **Varianz:**  Misst die Abweichung der Daten um ihren Mittelwert. Ein Streuungsmaß, welches die Verteilung von Werten um den Mittelwert kennzeichnet.

> **Entropie:** Eine Zunahme der Entropie bedeutet eine Zunahme der Unordnung im System.

> **Flächenverhältnis:** Der Prozentsatz des gesamten Bildes eines bestimmten Typs.

### Parameter I: Mittelwert: 
**Datenerfassung (grafische Darstellung) und Datenanalyse:** Wenn der Ausreißer durch grafische Darstellung bestimmt werden kann, sollte der kritische Punkt des Ausreißers als Schwellenwert für die Filterung verwendet werden. Wenn es keine Ausreißer oder kein Sinn der Ausreißer gibt, kann ein "binärer Baum" für eine binäre Auswahl verwendet werden, bis der Schwellenwert bestimmt ist.

Der Mittelwert aller Bilder wird ausgelesen und bildweise angezeigt und analysiert.

In [ ]:
frames_dir = "./extracted_frames_no_duplicates/"
meanmatrix=[]

for image_path in tqdm(glob.iglob(f"{frames_dir}*.jpg")):
    image = cv2.imread(image_path)
    #if not is_greyscale(image_path):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    mean = np.mean(image)
    meanmatrix = np.append(meanmatrix,mean)

Hier stellt zwei Hypothesen: Grenzwerte von Ausreißer und Grenzwerte von Unterbelichtung.

**Hypothese I-Ausreißer** Je nach Ausreißerbereich, Mittelwert größer als **82.32**

In [ ]:
os.chdir("/Users/xuhao/AUT_idee_BoxPlot/extracted_frames_no_duplicates")
frames_dir = "../extracted_frames_no_duplicates/"
ausreisserMittelwert = "./ausreisserMittelwert/"
counter = 0

for image_path in tqdm(glob.iglob(f"{frames_dir}*.jpg")):
    image = cv2.imread(image_path)
    #if not is_greyscale(image_path):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    varianz =np.mean(image)
    if varianz >= 82.32:
        os.rename(image_path, f"{ausreisserMittelwert}{image_path.lstrip(frames_dir)}")
        counter += 1

print(f"Anzahl der Ausreißer: {counter}")

2656 Bilder wurden entfernt, die unsaubere Bilder (viele Rauchwolken, Bildrauschen, lokale Überbelichtung) und globale Überbelichtung enthielten. Da es sich bei allen um Bilder handelt, die gelöscht werden müssen, wird die Schwellenwertauswahl nicht fortgesetzt.
Analyse: An dieser Stelle ist der Rest des Bildsatzes deutlich verbessert, von sauberen Bildern (nur Schmelzbad und Lichtbogen und Schmelzbad) bis hin zu unsauberen Bildern (unterbelichtet, mittlere Rauchwolken, hohes Grau durch viel Rauch).

In [4]:
schwellenwert_ausreisser=82.32

**Hypothese II: Schwellenwert der Unterbelichtung**

**"binary tree":**

**1. Versuch** für Bestimmung der Schwellenwerte der Unterbelichtung.
Hier verwendet "binary tree", d.h. kleiner als und gleich 82.32/2 ≈ 42

Theoretisch muss es höchstens ausgeführt werden: log2(256), dh 8 mal, und schon ist die richtige Schwelle gefunden.

In [ ]:
os.chdir("/Users/xuhao/AUT_idee_BoxPlot/extracted_frames_no_duplicates")
frames_dir = "../extracted_frames_no_duplicates/"
unterbelichtung_gemischt = "./unterbelichtung_gemischt/"
counter = 0

for image_path in tqdm(glob.iglob(f"{frames_dir}*.jpg")):
    image = cv2.imread(image_path)
    #if not is_greyscale(image_path):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    mid =np.mean(image)
    if mid <= 42:
        os.rename(image_path, f"{unterbelichtung_gemischt}{image_path.lstrip(frames_dir)}")
        counter += 1

print(f"Anzahl der unterbelichteten Bilder(gemischt mit Schmelzbad): {counter}")

Bei **1.Versuch** sind es Bilder mit "Lichtbogen" an der Unterbelichtung angehängt, also weiter zu "Binärbaum", das **2.Versuch** ist 42/2=21. 

Bei **2.Versuch** zeigt gut Ergebnisse, aber gemischt saubere Bilder(Bild nur mit Schmlezbad), es werden später durch Parameter "Varianz" filten.

Dann kann hier zeigen, Schwellenwerte der Mittelwert von Unterbelichtung ist **21 Pixelwert**.

In [5]:
Schwellenwert_Unterbelichtung= 21

### Parameter II: Varianz 

#### Hypothese III: Ausreißer
**Datenerfassung (grafische Darstellung) und Datenanalyse:**

Der nächste Schritt besteht darin, die Unterbelichtung der Mischung zu “bereinigen (das Bild mit dem Schmelzbad zu entfernen)”.

hier verwendet **Varianz**, werden beide deutlich trennen. 

**Datenanalyse:** Erst check die Ausreißer Punkte.

Versuchsdaten aufzeichnen:
> Erste Versuche: 1756 -> keine Schwellenwert.

> Zweite Versuche:(np.max(varianzmatrix)(3714.208183737658 )- 1756)/2 + 1756 = 2735 -> keine Schwellenwert.

> Dritte Versuche: 3224

In [ ]:
os.chdir("/Users/xuhao/AUT_idee_BoxPlot/extracted_frames_no_duplicates")
frames_dir = "../extracted_frames_no_duplicates/"
ausreisserVar = "./ausreisserVar/"
counter = 0

for image_path in tqdm(glob.iglob(f"{frames_dir}*.jpg")):
    image = cv2.imread(image_path)
    #if not is_greyscale(image_path):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    varianz =np.var(image)
    if varianz > 3224:
        os.rename(image_path, f"{ausreisserVar}{image_path.lstrip(frames_dir)}")
        counter += 1

print(f"Anzahl der ausreisser Varianz: {counter}")

Die Ausreißer hier sind nicht vollständig nicht verfügbar, daher wird **"binary tree"** verwendet, um den grafischen Schwellenwert eines bestimmten Fehlers auszuwählen:

#### Hypothese IV:  "binary tree zur Kontrastarmes Bild:" 

> Erste Versuch：lower_quartile ≈ 340 Varianz

> Zweite Versuch：lower_quartile / 2  = 170 Varianz

> Zweite Versuch：170 / 2  = 85 Varianz

In [ ]:
frames_dir = "../extracted_frames_no_duplicates/"
ausreisserVar = "./ausreisserVar/"
counter = 0

for image_path in tqdm(glob.iglob(f"{frames_dir}*.jpg")):
    image = cv2.imread(image_path)
    #if not is_greyscale(image_path):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    varianz =np.var(image)
    if varianz <= 85:
        os.rename(image_path, f"{ausreisserVar}{image_path.lstrip(frames_dir)}")
        counter += 1

print(f"Anzahl der ausreisser Varianz: {counter}")

In [ ]:
schwellenwert_varianz=85

[Top of Notebook](#divtop)

<a name="pipeline"></a>
## Die gesamte Pipeline ausführen

<a name="step1"></a>
### read step

Interactive window (input parameters)

In [ ]:
# Video to image
os.chdir("/Users/xuhao/AUT_idee_BoxPlot") 
file_name_pattern = f"{os.getcwd()}/Sample*/**/*.avi" 
video_files = glob.glob(file_name_pattern, recursive=True); len(video_files)

# Calculate the number of frames
file_name_pattern_num = f"{os.getcwd()}/Sample1_Video_220615-152743-STtoST_3_6_10_*.jpg"
filename = './outputdata/220615-152743-STtoST_3_6_10.avi'

Laufen die Funktion. Hier wartet vielleicht von 5 bis 10 min.

In [ ]:
getVideoToImage(file_name_pattern, video_files)

Check die eigentliche Framesrate des Videos:

    Pure Framerate = num_image / seconds

In [ ]:
fps  = get_number_from_image(file_name_pattern_num) / get_duration_from_cv2(filename)
print("Eigentlich framerate : {0}".format(fps))

[Top of Notebook](#divtop)

<a name="step2"></a>
### data clean step

Interactive window (input parameters)

In [ ]:
frames_dir = "../extracted_frames_no_duplicates/"
removed_frames_dir_over = "./removed_frames_overexposed/"
removed_frames_dir_dirty = "./removed_frames_dirty/"
removed_frames_dir_unter = "./removed_frames_unterexposed/"

removed_counter_dirty = 0
removed_counter_under = 0
removed_counter_over  = 0

# Darstellung der Schwellenwerte:
schwellenwert_mittelwert= 82.32
schwellenwert_mittelwert_unterbelichtet = 21
schwellenwert_varianz= 85

# Eingabe der erwartet Flächenprozent:
expect_highsmoke = 0.088
expect_midsmoke =0.20

**Backbone des Systemes:**


    For image_path in Ordner:
        image = cv2.imread(image_path)
        if not is_greyscale(image): 
           image = RGB_to_YUV(image)  
        
        mittelwerte =np.mean(image)
        varianz = np.var(image)
           
        if mittelwerte < schwellenwerte_unterbelichtung(21 Grauwert):      #________1.Stufe
           
           if is_underexposed(image):            #_________________________________ 2.Stufe 
           - remove(image_path, "Unterbelichtung")                       
           else:
           - keep on
           
        else if  mittelwerte < schwellenwerte_mittelwert(82.32 Grauwerte):   #_____ 1.Stufe
           
           if high_smoke(image)>expect_highsmoke(0.0088)    
              or mid_smoke(image)>expect_highsmoke(0.20) : #_______________ 2.Stufe
           - remove(image_path, "unsauber")
           
           else if varianz < Schwellenwert_varianz: 
           - remove(image_path, "unsauber")
           
           else:
           - keep on
        
        else:               #_______________________________________________________1.Stufe: 
           
           if is_unterexposed(image):         #____________________________________ 2.Stufe                   
           - remove(image_path, "Überbelichtung")
           else:
           - remove(image_path, "unsauber")

Durch parallel Weg wird auch die Rechenaufwand Signifikant reduziert.

In [ ]:
for image_path in tqdm(glob.iglob(f"{frames_dir}*.jpg")):
    image = cv2.imread(image_path)
    #if not is_greyscale(image_path):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    mittelwerte =np.mean(image)
    varianz = np.var(image)
    
    if mittelwerte < schwellenwert_mittelwert_unterbelichtet:
        if is_underexposed(varianz,schwellenwert_varianz):
            os.rename(image_path, f"{removed_frames_dir_unter}{image_path.lstrip(frames_dir)}")
            removed_counter_under += 1
    
    elif schwellenwert_mittelwert_unterbelichtet < mittelwerte < schwellenwert_mittelwert:
        if varianz < schwellenwert_varianz: 
            os.rename(image_path, f"{removed_frames_dir_dirty}{image_path.lstrip(frames_dir)}")
            removed_counter_dirty += 1
            
        elif high_smoke(image) > expect_highsmoke or mid_smoke(image) > expect_midsmoke:
            os.rename(image_path, f"{removed_frames_dir_dirty}{image_path.lstrip(frames_dir)}")
            removed_counter_dirty += 1
    
    elif mittelwerte > schwellenwert_mittelwert:
        if is_overexposed(image):
            os.rename(image_path, f"{removed_frames_dir_over}{image_path.lstrip(frames_dir)}")
            removed_counter_over += 1
            
        else:
            os.rename(image_path, f"{removed_frames_dir_dirty}{image_path.lstrip(frames_dir)}")
            removed_counter_dirty += 1
            
removed_counter = removed_counter_dirty+removed_counter_over+removed_counter_under
print(f"Number of frames that have been removed from the dirty data: {removed_counter_dirty}")
print(f"Number of frames that have been removed from the overexposed data: {removed_counter_over}")
print(f"Number of frames that have been removed from the underexposed data: {removed_counter_under}")
print(f"Number of frames that have been removed from the all data: {removed_counter}")

[Top of Notebook](#divtop)

<a name="datensatz"></a>
## Datensatz
Datensatz wird in [TUBcloud](https://tubcloud.tu-berlin.de/s/m2YseaFsLzxteoY?path=%2F06%20Daten) festgelegt. 
Name der Ordnern:  
> - Alle **sauber Bildern** speichern unten Ordner "extracted_frames_no_duplicates"
> - Alle **überbelichtete Bildern** speichern unten  Ordner "removed_frames_overexposed"
> - Alle **unterbelichtete Bildern** speichern unten  Ordner "removed_frames_unterexposed"
> - Alle **unsauber Bildern** speichern unten  Ordner "removed_frames_unsauber"

**Pipeline** /Workflow: HTML und Notebook in [Github](https://git.tu-berlin.de/iat/aut-project-gefertec/-/tree/Bildvorverarbeitung).

[Top of Notebook](#divtop)

<a name="resultat"></a>
## Resultat

**Videos zu Image per Frames**
> - Es wurde realisiert, das Video entsprechend den Frames in fortlaufende Bilder umzuwandeln
> - Die erzeugten fortlaufenden Bilder werden automatisch in einem Ordner mit dem Namen "extracted_frames_no_duplicates" abgelegt und hinzugefügt
> - Funktion zur Erkennung der eigentlichen Framerate implementiert

**Data Clean**

Finally, "Data Clean" erreicht...
> + ... Automatisierter Prozess
> + ... Die aktuell angezeigten Bilder sind in drei Typen unterteilt: Überlichtet, Unterlichtet und Unsauber Bild
> + ... Ca. 93% der Bilder mit Grauwert von 0 bis 100. So außer der überbelichtete und unterbelichtet Bilder ist noch ca. 90 % Bilder machbar.
> + ... Ca. 11.8 % (3188/27012) der Bilder wird rausgeschmissen. 
> + ... Überbelichtung:100% richtig gefiltert, besitzt 0.4% (103/27012)
> + ... Unterbelichtung: 100% richtig gefiltert, besitzt 0.6% (160/27012)
> + ... Unsaubere Bilder: Wegen der Grauwert-Konflikte, die Modell NICHT total robust. Aber Schmutzige Bilder werden jedoch weitgehend gefiltert; Möglichkeit für qualitativ hochwertige Bilder (Muss eine bestimmte Menge an sauberen Bildern verschwenden. Erste Sample als Beispiel, die Verschwendungsrate beträgt 36% (18/50), und besitzen 0.5% (18/3773) des 1. Simple )

[Top of Notebook](#divtop)